In [ ]:
import pandas as pd 
import os 
import sys

sys.path.append("../src")
from llm_client import call_llm
from prompt_templates import *

c:\Stellantis\HomeNestReviews_llm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Stellantis\HomeNestReviews_llm\notebooks\../src\llm_client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
df = pd.read_csv("../data/raw/homenest_reviews.csv")
df.head()

,review_id,product_category,product_name,review_text,star_rating,review_date,verified_purchase
0,1,Bedroom Furniture,DreamRest Memory Foam Mattress,My DreamRest Memory Foam Mattress arrived well...,5,2024-02-08,False
1,2,Kitchen Appliances,InstaBrew Coffee Maker,The InstaBrew Coffee Maker itself is fine but ...,2,2023-09-11,False
2,3,Living Room,GlowLight Floor Lamp,My GlowLight Floor Lamp arrived well packaged ...,5,2023-07-03,False
3,4,Living Room,CozySofa L-Shape Sectional,The CozySofa L-Shape Sectional is okay. Does w...,3,2023-06-30,True
4,5,Kitchen Appliances,TurboBlend Pro 5000,Delivery took 3 weeks and the TurboBlend Pro 5...,3,2024-09-23,True


In [11]:
def rating_to_sentiment(star):
    if star > 4 :
        return "Positive"
    elif star == 3:
        return "Neutral"
    else :
        return "Negative"
    


df["ground_truth"] = df["star_rating"].apply(rating_to_sentiment)

Zero -Shot and Few Shot prompts

In [8]:
ZERO_SHOT_SYSTEM = """
You are a sentiment classifier.

Respond ONLY with:
Positive
Negative
Neutral
"""

In [9]:
FEW_SHOT_SYSTEM = "You are a sentiment classifier."

In [ ]:
results = []

for i,row in df.head(200).iterrows():

    review = row["review_text"]
    truth = row["ground_truth"]

    zero = call_llm(ZERO_SHOT_SYSTEM, f"Review: {review}", provider="mistral")

    few_prompt = f"""
Classify the sentiment of customer reviews.

Examples:
Review: 'Absolutely love this blender! Works perfectly.' -> Positive
Review: 'Broke after 2 weeks. Total waste of money.' -> Negative
Review: 'It is okay. Does what it says, nothing special.' -> Neutral

Now classify:

Review: '{review}' ->
"""

    few = call_llm(FEW_SHOT_SYSTEM, few_prompt, provider="mistral")

    results.append({
        "review_id": i,
        "ground_truth": truth,
        "zero_shot": zero.strip(),
        "few_shot": few.strip()
    })

In [13]:
print(results)

[{'review_id': 0, 'ground_truth': 'Positive', 'zero_shot': 'Positive', 'few_shot': 'Positive'}, {'review_id': 1, 'ground_truth': 'Negative', 'zero_shot': 'Neutral', 'few_shot': 'Neutral (leaning towards Negative)\n\nThe review expresses a mixed sentiment. The customer states that the product itself is "fine," which is a neutral to slightly positive statement. However, the negative aspect regarding the destroyed packaging brings the overall sentiment closer to neutral, with a slight lean towards negative due to the frustration implied.'}, {'review_id': 2, 'ground_truth': 'Positive', 'zero_shot': 'Positive', 'few_shot': 'Positive'}, {'review_id': 3, 'ground_truth': 'Neutral', 'zero_shot': 'Neutral', 'few_shot': 'Neutral'}, {'review_id': 4, 'ground_truth': 'Neutral', 'zero_shot': 'Negative', 'few_shot': 'Negative\n\nThe review expresses dissatisfaction with the delivery time and the condition of the packaging, which are significant issues. Although the product itself was okay, the negativ

In [14]:
results_df = pd.DataFrame(results)
results_df

,review_id,ground_truth,zero_shot,few_shot
0,0,Positive,Positive,Positive
1,1,Negative,Neutral,Neutral (leaning towards Negative)\n\nThe revi...
2,2,Positive,Positive,Positive
3,3,Neutral,Neutral,Neutral
4,4,Neutral,Negative,Negative\n\nThe review expresses dissatisfacti...
5,5,Positive,Positive,Positive
6,6,Neutral,Neutral,Neutral
7,7,Negative,Positive,Positive
8,8,Positive,Positive,Positive
9,9,Positive,Negative,Negative


In [15]:
results_df["zero_correct"] = results_df["ground_truth"] == results_df["zero_shot"]
results_df["few_correct"] = results_df["ground_truth"] == results_df["few_shot"]

results_df

,review_id,ground_truth,zero_shot,few_shot,zero_correct,few_correct
0,0,Positive,Positive,Positive,True,True
1,1,Negative,Neutral,Neutral (leaning towards Negative)\n\nThe revi...,False,False
2,2,Positive,Positive,Positive,True,True
3,3,Neutral,Neutral,Neutral,True,True
4,4,Neutral,Negative,Negative\n\nThe review expresses dissatisfacti...,False,False
5,5,Positive,Positive,Positive,True,True
6,6,Neutral,Neutral,Neutral,True,True
7,7,Negative,Positive,Positive,False,False
8,8,Positive,Positive,Positive,True,True
9,9,Positive,Negative,Negative,False,False


In [19]:
print("Zero-shot prompt accuracy :",results_df["zero_correct"].value_counts())
print("\n\nFew-shots prompt accuracy :",results_df["few_correct"].value_counts())

Zero-shot prompt accuracy : zero_correct
True     14
False     6
Name: count, dtype: int64


Few-shots prompt accuracy : few_correct
True     14
False     6
Name: count, dtype: int64


In [20]:
zero_acc = results_df["zero_correct"].mean()
few_acc = results_df["few_correct"].mean()

print("Zero-shot accuracy:", zero_acc)
print("Few-shot accuracy:", few_acc)

Zero-shot accuracy: 0.7
Few-shot accuracy: 0.7


Observation : Mistral is Returning Sentences for few cases instead of Lableing 

Chain of Thought for Nuanced Reviews 

In [23]:
# Find the reviews where Zero-shot Failed 

difficult_reviews = results_df[results_df['zero_correct'] == False]
difficult_reviews.head(10)

,review_id,ground_truth,zero_shot,few_shot,zero_correct,few_correct
1,1,Negative,Neutral,Neutral (leaning towards Negative)\n\nThe revi...,False,False
4,4,Neutral,Negative,Negative\n\nThe review expresses dissatisfacti...,False,False
7,7,Negative,Positive,Positive,False,False
9,9,Positive,Negative,Negative,False,False
17,17,Negative,Positive,Positive,False,False
18,18,Negative,Positive,Positive,False,False


In [ ]:
difficult_reviews = difficult_reviews.merge(
    df[["review_text"]],
    left_on="review_id",
    right_index=True
)

In [ ]:
COT_SYSTEM = """You are an expert review analyst.
 Think carefully before classifying.
 
 USER: Analyze this customer review step by step:
 1. What is the customer's main experience (positive or negative)?
 2. Are there any contradictions or mixed feelings?
 3. What is the dominant sentiment overall?
 4. Final classification: Positive, Negative, or NeutralReview: '{review_text}'
 
 Respond in this format:
 Step 1: [your analysis]
 Step 2: [your analysis]
 Step 3: [your analysis]
 Classification: [Positive/Negative/Neutral] """